# Separating ambiguity from ordinary risk

## Risk Inside One Probability Law

Let the possible returns be \(x_1,\ldots,x_K\), and let one probability law assign probability \(p_k\) to outcome \(x_k\). Its mean return is

$$
\mu=\sum_{k=1}^{K}p_kx_k,
$$

and its return variance is

$$
\sigma^2=\sum_{k=1}^{K}p_k(x_k-\mu)^2.
$$

Here, \(K\) is the number of mutually exclusive outcomes; \(x_k\) is outcome \(k\); \(p_k\) is its probability; \(\mu\) is the probability weighted mean; and \(\sigma^2\) measures dispersion of outcomes around that mean. Risk is present when one known law assigns positive probability to different outcomes.

## Ambiguity Across Probability Laws

Suppose there are \(J\) possible probability laws. Law \(j\) assigns probability \(p_{j,k}\) to outcome \(k\), and the law receives prior weight \(q_j\). The mean probability of outcome \(k\) is

$$
\bar p_k=\sum_{j=1}^{J}q_jp_{j,k},
$$

and the population variance of that probability is

$$
v_k=\sum_{j=1}^{J}q_j(p_{j,k}-\bar p_k)^2.
$$

Here, \(J\) is the number of supplied laws; \(q_j\geq0\) is the weight on law \(j\), with \(\sum_jq_j=1\); \(p_{j,k}\) is the probability of outcome \(k\) under law \(j\); \(\bar p_k\) is the prior weighted mean probability; and \(v_k\) measures disagreement about that probability. Ambiguity concerns variation across the laws, not variation in the outcomes under one law.

In [1]:
from pathlib import Path
import calendar
import hashlib
import sys

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import HTML, display as _display


def display(value):
    module = value.__class__.__module__
    if module.startswith("pandas") and hasattr(value, "_repr_html_"):
        html = value._repr_html_()
        return _display(HTML('<div style="display:flex;justify-content:center;">' + html + "</div>"))
    return _display(value)
from scipy.stats import norm


BOOK_ROOT = Path.cwd().parents[1]
if str(BOOK_ROOT) not in sys.path:
    sys.path.insert(0, str(BOOK_ROOT))
from book_support import ensure_dataset

import RiskLabAI
from RiskLabAI.features.entropy_features.ambiguity import normal_prior_bin_probabilities, probability_ambiguity

assert RiskLabAI.__version__ == "3.2.0"

DATA_PATH = ensure_dataset(BOOK_ROOT, "fred_nasdaq")
FIGURES = BOOK_ROOT / "outputs" / "figures" / "04_probability_ambiguity"
TABLES = BOOK_ROOT / "outputs" / "tables" / "04_probability_ambiguity"
FIGURE_DATA = BOOK_ROOT / "outputs" / "figure_data" / "04_probability_ambiguity"
for directory in [FIGURES, TABLES, FIGURE_DATA]:
    directory.mkdir(parents=True, exist_ok=True)

PALETTE = {
    "background": "#FFFFFF",
    "axes": "#FFFFFF",
    "text": "#1B365D",
    "cyan": "#0072B2",
    "amber": "#C76D00",
    "green": "#00866A",
    "rose": "#B83B3B",
    "violet": "#A03A78",
    "grid": "#D9DEE5",
}
plt.rcParams.update({
    "font.family": "DejaVu Serif",
    "font.size": 11.5,
    "figure.facecolor": PALETTE["background"],
    "axes.facecolor": PALETTE["axes"],
    "axes.edgecolor": PALETTE["text"],
    "axes.labelcolor": PALETTE["text"],
    "axes.titlecolor": PALETTE["text"],
    "xtick.color": PALETTE["text"],
    "ytick.color": PALETTE["text"],
    "text.color": PALETTE["text"],
    "grid.color": PALETTE["grid"],
    "grid.linestyle": ":",
    "legend.facecolor": PALETTE["axes"],
    "legend.edgecolor": PALETTE["grid"],
    "savefig.facecolor": PALETTE["background"],
    "savefig.transparent": False,
    "savefig.dpi": 300,
})


def sha256(path):
    digest = hashlib.sha256()
    with path.open("rb") as stream:
        for block in iter(lambda: stream.read(1024 * 1024), b""):
            digest.update(block)
    return digest.hexdigest().upper()


def rendered_value(value):
    if pd.isna(value):
        return ""
    if isinstance(value, (pd.Timestamp, np.datetime64)):
        return pd.Timestamp(value).date().isoformat()
    if isinstance(value, (int, np.integer)):
        return f"{int(value):,}"
    if isinstance(value, (float, np.floating)):
        number = float(value)
        if number == 0:
            return "0"
        if number.is_integer():
            return f"{int(number):,}"
        magnitude = abs(number)
        if magnitude < 0.0001 or magnitude >= 1000000:
            return f"{number:.6g}"
        return f"{number:.6f}".rstrip("0").rstrip(".")
    return str(value)


def rendered_frame(frame):
    shown = frame.apply(lambda column: column.map(rendered_value))
    for column in frame.columns:
        if "year" in str(column).lower():
            shown[column] = frame[column].map(lambda value: "" if pd.isna(value) else str(int(value)) if isinstance(value, (int, float, np.integer, np.floating)) and float(value).is_integer() else str(value))
    return shown


def save_table(frame, stem, caption, label, fit=False):
    shown = rendered_frame(frame)
    shown.to_csv(TABLES / f"{stem}.csv", index=False)
    latex = shown.to_latex(index=False, escape=True, caption=caption, label=label, position="htbp")
    if fit:
        latex = latex.replace("\\begin{tabular}", "\\begin{adjustbox}{max width=\\textwidth}\\begin{tabular}", 1).replace("\\end{tabular}", "\\end{tabular}\\end{adjustbox}", 1)
    (TABLES / f"{stem}.tex").write_text(latex, encoding="utf-8")
    display(shown)
    return shown


def finish_axis(axis, grid=True):
    axis.spines["top"].set_visible(False)
    axis.spines["right"].set_visible(False)
    if grid:
        axis.grid(True, alpha=0.65)
    axis.set_axisbelow(True)


def save_figure(figure, stem):
    figure.savefig(FIGURES / f"{stem}.png", dpi=300, bbox_inches="tight", facecolor=PALETTE["background"], transparent=False)
    figure.savefig(FIGURES / f"{stem}.pdf", bbox_inches="tight", facecolor=PALETTE["background"], transparent=False)


def load_public_data():
    expected_hash = "2E988948FFCB108DA4902DFEDAA5DF756E7E102DC523A99A8916A0533291F2D7"
    assert DATA_PATH.is_file()
    assert sha256(DATA_PATH) == expected_hash
    raw = pd.read_csv(DATA_PATH)
    raw["Date"] = pd.to_datetime(raw["observation_date"])
    raw["NASDAQ Composite"] = pd.to_numeric(raw["NASDAQCOM"], errors="coerce")
    data = raw.dropna(subset=["NASDAQ Composite"]).sort_values("Date").reset_index(drop=True)
    data["Daily log return"] = np.log(data["NASDAQ Composite"] / data["NASDAQ Composite"].shift(1))
    data["Year"] = data["Date"].dt.year
    data["Month"] = data["Date"].dt.month
    return data


def monthly_moments(data):
    returns = data.dropna(subset=["Daily log return"]).copy()
    moments = returns.groupby(["Year", "Month"])["Daily log return"].agg(
        Mean="mean",
        Standard_deviation=lambda values: values.std(ddof=0),
        Observations="size",
    ).reset_index()
    return moments


def normal_edges(width):
    intervals = int(round(0.12 / width))
    return np.linspace(-0.06, 0.06, intervals + 1)


def bin_labels(edges):
    labels = [f"Below {100 * edges[0]:g}%"]
    labels.extend(f"{100 * left:g}% to {100 * right:g}%" for left, right in zip(edges[:-1], edges[1:]))
    labels.append(f"Above {100 * edges[-1]:g}%")
    return labels

In [2]:
outcomes = np.array([-2, -1, 0, 1, 2], dtype=float)
known_narrow = np.tile([0.05, 0.15, 0.60, 0.15, 0.05], (3, 1))
known_wide = np.tile([0.30, 0.15, 0.10, 0.15, 0.30], (3, 1))
shifting_center = np.array([
    [0.55, 0.30, 0.10, 0.04, 0.01],
    [0.05, 0.15, 0.60, 0.15, 0.05],
    [0.01, 0.04, 0.10, 0.30, 0.55],
])
changing_tails = np.array([
    [0.05, 0.15, 0.60, 0.15, 0.05],
    [0.20, 0.25, 0.10, 0.25, 0.20],
    [0.45, 0.05, 0.00, 0.05, 0.45],
])
weights = np.full(3, 1 / 3)

prior_table = pd.DataFrame(changing_tails, columns=[f"Return {value:g}" for value in outcomes])
prior_table.insert(0, "Supplied law", ["Law 1", "Law 2", "Law 3"])
save_table(prior_table, "discrete_prior_example", "Three supplied probability laws over five return outcomes.", "tab:chapter5-discrete-prior-example", fit=True)

case_rows = []
for name, probabilities in [
    ("Known narrow law", known_narrow),
    ("Known wide law", known_wide),
    ("Changing center", shifting_center),
    ("Changing tails", changing_tails),
]:
    mean_probability = weights @ probabilities
    expected_return = float(mean_probability @ outcomes)
    expected_return = 0.0 if abs(expected_return) < 1e-12 else expected_return
    return_variance = mean_probability @ ((outcomes - expected_return) ** 2)
    ambiguity = probability_ambiguity(probabilities, prior_weights=weights)["unscaled"]
    case_rows.append({
        "Case": name,
        "Mean return": expected_return,
        "Return variance": return_variance,
        "Probability ambiguity": ambiguity,
    })
case_frame = pd.DataFrame(case_rows)
save_table(case_frame, "risk_ambiguity_cases", "Return risk and probability ambiguity in four supplied examples.", "tab:chapter5-risk-ambiguity-cases")

law_rows = []
for law_index, probabilities in enumerate(changing_tails, start=1):
    for outcome, probability in zip(outcomes, probabilities):
        law_rows.append({"Supplied law": f"Law {law_index}", "Return outcome": outcome, "Probability": probability})
law_frame = pd.DataFrame(law_rows)
law_frame.to_csv(FIGURE_DATA / "supplied_probability_laws.csv", index=False)
case_frame.to_csv(FIGURE_DATA / "risk_ambiguity_cases.csv", index=False)

,Supplied law,Return -2,Return -1,Return 0,Return 1,Return 2
0,Law 1,0.05,0.15,0.6,0.15,0.05
1,Law 2,0.2,0.25,0.1,0.25,0.2
2,Law 3,0.45,0.05,0,0.05,0.45


,Case,Mean return,Return variance,Probability ambiguity
0,Known narrow law,0,0.7,0
1,Known wide law,0,2.7,0
2,Changing center,0,1.953333,0.043069
3,Changing tails,0,2.166667,0.030778


In [3]:
figure, axis = plt.subplots(figsize=(9.6, 5.5), constrained_layout=True)
colors = [PALETTE["cyan"], PALETTE["amber"], PALETTE["green"]]
markers = ["o", "s", "^"]
for law_index, color, marker in zip(range(1, 4), colors, markers):
    part = law_frame[law_frame["Supplied law"] == f"Law {law_index}"]
    axis.plot(part["Return outcome"], part["Probability"], color=color, marker=marker, linewidth=2.1, markersize=7, label=f"Law {law_index}")
axis.set_xlabel("Return outcome")
axis.set_ylabel("Probability")
axis.set_title("The Same Outcome Can Receive Different Probabilities")
axis.set_xticks(outcomes)
axis.legend()
finish_axis(axis)
save_figure(figure, "supplied_probability_laws")
display(figure)
plt.close(figure)

<Figure size 960x550 with 1 Axes>

In [4]:
figure, axis = plt.subplots(figsize=(9.6, 5.6), constrained_layout=True)
colors = [PALETTE["cyan"], PALETTE["amber"], PALETTE["green"], PALETTE["rose"]]
for (_, row), color in zip(case_frame.iterrows(), colors):
    axis.scatter(row["Return variance"], row["Probability ambiguity"], s=115, color=color)
    axis.annotate(row["Case"], (row["Return variance"], row["Probability ambiguity"]), xytext=(7, 7), textcoords="offset points")
axis.set_xlabel("Return variance under the mean law")
axis.set_ylabel("Probability ambiguity")
axis.set_title("Risk and Ambiguity Are Different Summaries")
finish_axis(axis)
save_figure(figure, "risk_and_ambiguity_examples")
display(figure)
plt.close(figure)

<Figure size 960x560 with 1 Axes>

## Verifying the finite ambiguity measure

The finite supplied-prior ambiguity measure is

$$
A=\sum_{k=1}^{K}\bar p_kv_k.
$$

Here, \(A\) is the unscaled ambiguity measure; \(\bar p_k\) is the mean probability of outcome bin \(k\); and \(v_k\) is the variance of that probability across supplied laws. Every term is nonnegative. The measure is therefore nonnegative, and it is zero when all positively weighted laws agree on every bin.

The variance in each bin also has the pairwise form

$$
v_k=\frac{1}{2}\sum_{i=1}^{J}\sum_{j=1}^{J}q_iq_j(p_{i,k}-p_{j,k})^2.
$$

Here, \(i\) and \(j\) identify two supplied laws. The squared difference records their disagreement in bin \(k\), while \(q_iq_j/2\) supplies the pair weight without double counting. This identity provides an independent calculation of the measure.

In [5]:
probabilities = np.array([[0.2, 0.8], [0.6, 0.4], [1.0, 0.0]])
prior_weights = np.array([0.2, 0.3, 0.5])
result = probability_ambiguity(probabilities, prior_weights=prior_weights, bin_width=0.002)
mean_probability = prior_weights @ probabilities
direct_variance = prior_weights @ ((probabilities - mean_probability) ** 2)
direct_value = float(mean_probability @ direct_variance)
pairwise_variance = np.array([
    0.5 * sum(prior_weights[i] * prior_weights[j] * (probabilities[i, k] - probabilities[j, k]) ** 2 for i in range(3) for j in range(3))
    for k in range(2)
])
pairwise_value = float(mean_probability @ pairwise_variance)
permuted_value = probability_ambiguity(probabilities[::-1], prior_weights=prior_weights[::-1])["unscaled"]
identical_value = probability_ambiguity([[0.5, 0.5]] * 3)["unscaled"]
extreme_value = probability_ambiguity([[0, 1], [1, 0]])["unscaled"]

checks = pd.DataFrame({
    "Check": [
        "Library and direct enumeration agree",
        "Library and pairwise identity agree",
        "Prior ordering leaves the value unchanged",
        "Identical laws give zero ambiguity",
        "Two opposite certain laws give one quarter",
        "Scaled value follows the stated relation",
    ],
    "Calculated difference": [
        abs(result["unscaled"] - direct_value),
        abs(result["unscaled"] - pairwise_value),
        abs(result["unscaled"] - permuted_value),
        abs(identical_value),
        abs(extreme_value - 0.25),
        abs(result["scaled"] - result["unscaled"] / (0.002 * 0.998)),
    ],
    "Result": ["Pass"] * 6,
})
assert np.all(checks["Calculated difference"] < 1e-12)
save_table(checks, "finite_formula_checks", "Independent checks of the finite ambiguity formula.", "tab:chapter5-finite-formula-checks", fit=True)

boundaries = pd.DataFrame({
    "Case": ["One known wide law", "Three identical laws", "Two opposite certain laws"],
    "Return variance": [known_wide[0] @ outcomes ** 2, known_narrow[0] @ outcomes ** 2, 1],
    "Probability ambiguity": [
        probability_ambiguity(known_wide[:1])["unscaled"],
        identical_value,
        extreme_value,
    ],
})
save_table(boundaries, "ambiguity_boundary_cases", "Boundary cases that separate return dispersion from probability disagreement.", "tab:chapter5-ambiguity-boundary-cases")

,Check,Calculated difference,Result
0,Library and direct enumeration agree,0,Pass
1,Library and pairwise identity agree,1.38778e-17,Pass
2,Prior ordering leaves the value unchanged,0,Pass
3,Identical laws give zero ambiguity,0,Pass
4,Two opposite certain laws give one quarter,0,Pass
5,Scaled value follows the stated relation,0,Pass


,Case,Return variance,Probability ambiguity
0,One known wide law,2.7,0
1,Three identical laws,0.7,0
2,Two opposite certain laws,1,0.25


,Case,Return variance,Probability ambiguity
0,One known wide law,2.7,0
1,Three identical laws,0.7,0
2,Two opposite certain laws,1,0.25
